In [ ]:
import pandas as pd, numpy as np, os
from vivarium_gbd_access.shared_functions import get_population, get_covariate_estimates, get_ids
import vivarium.artifact as artifact
from vivarium_gbd_access.shared_functions import get_machinery_estimates

from lsff_utils import paths

In [ ]:
location = 'nigeria'
vehicle = 'rice'
data_source = paths.data_source(default=paths.MODEL_NUMBER)  # or "local" for this working tree

In [ ]:
l = get_ids('location')
location_id = l.loc[l.location_name==location.title()].location_id.values[0]

# Resolved from data_source; the run is the one latest_run.txt names.
results_dir = paths.latest_results(
    paths.source_root(paths.MATERNAL_RESULTS_ROOT, data_source), location, vehicle
)
artifact_dir = paths.artifact_path(
    paths.source_root(paths.MATERNAL_ARTIFACT_ROOT, data_source), location, vehicle
)
print(f"{data_source} {vehicle}/{location}\n  results:  {results_dir}\n  artifact: {artifact_dir}")

# first just maternal outputs

In [ ]:
year_id=2023
age_group_ids = list(range(7,17))
asfr = get_covariate_estimates(covariate_id=13, 
                               location_id=location_id, 
                               age_group_id=age_group_ids,
                               sex_id=2,
                               release_id=16,
                               year_id=year_id).rename(columns={'mean_value':'val'})
sbr = get_covariate_estimates(covariate_id=2267, 
                               location_id=location_id, 
                               release_id=16,
                               year_id=year_id).rename(columns={'mean_value':'val'})
incidence_c995 = get_machinery_estimates(entity='cause',
                                         entity_id=995,
                                         release_id=16,
                                         year_id=2023,
                                         sex_id=2,
                                         location_id=location_id,
                                         age_group_id=age_group_ids,
                                         measure_id=6,
                                         metric_id=3
                                         ).rename(columns={'point_estimate':'val'})
incidence_c374 = get_machinery_estimates(entity='cause',
                                         entity_id=374,
                                         release_id=16,
                                         year_id=2023,
                                         sex_id=2,
                                         location_id=location_id,
                                         age_group_id=age_group_ids,
                                         measure_id=6,
                                         metric_id=3
                                         ).rename(columns={'point_estimate':'val'})
live_inc = asfr.set_index(['location_id','age_group_id'])[['val']]
still_inc = (asfr.set_index(['location_id','age_group_id'])[['val']]
               * sbr.set_index(['location_id'])[['val']])
other_inc = (incidence_c995.set_index(['location_id','age_group_id'])[['val']]
             + incidence_c374.set_index(['location_id','age_group_id'])[['val']])
preg_inc = (asfr.set_index(['location_id','age_group_id'])[['val']]
            + (asfr.set_index(['location_id','age_group_id'])[['val']]
               * sbr.set_index(['location_id'])[['val']])
            + incidence_c995.set_index(['location_id','age_group_id'])[['val']]
            + incidence_c374.set_index(['location_id','age_group_id'])[['val']])
frac_live = (live_inc / preg_inc).dropna()
frac_still = (still_inc / preg_inc).dropna()
frac_other = (other_inc / preg_inc).dropna()
frac_other.head()

In [ ]:
# load up pregnancy incidence
art = artifact.Artifact(artifact_dir, filter_terms=['sex==Female', 'age_start >= 10', 'age_end <=55'])
asfr = art.load('covariate.age_specific_fertility_rate.estimate')
sbr = art.load('covariate.stillbirth_28_weeks_to_live_birth_ratio.estimate')   # GBD 2023 split the covariate into 20/28-week variants; model 1.1 carries the 28-week one
inc_mis = art.load('cause.maternal_abortion_and_miscarriage.raw_incidence_rate')
inc_ect = art.load('cause.ectopic_pregnancy.raw_incidence_rate')
art_preg_inc = (asfr 
               + asfr * sbr.value.values[0]
               + inc_mis 
               + inc_ect)
trans = pd.read_parquet(paths.measure_path(results_dir, "transition_count_pregnancy"))
art_live_frac = (asfr / art_preg_inc)[[f'draw_{x}' for x in trans.input_draw.unique()]].mean(axis=1)
art_still_frac = ((asfr * sbr.value.values[0]) / art_preg_inc)[[f'draw_{x}' for x in trans.input_draw.unique()]].mean(axis=1)
art_other_frac = ((inc_mis + inc_ect) / art_preg_inc)[[f'draw_{x}' for x in trans.input_draw.unique()]].mean(axis=1)
art_other_frac.head()

In [ ]:
# are birth outcomes being counted every timestep or once per sim?
outcomes = pd.read_parquet(paths.measure_path(results_dir, "pregnancy_outcome_count"))
outcomes = outcomes.groupby(['scenario','input_draw'], observed=False)['value'].sum() / 10_000_000 # population size per draw... note this could/should be updated to be read in from the model spec/branches file
assert np.all(outcomes == 1), "Count of pregnancy outcomes does not correspond to expected population size"
# looks great! now being counted once in the same way that the intervention counts are below

In [ ]:
cols = ['scenario','age_group','input_draw']
outcomes = pd.read_parquet(paths.measure_path(results_dir, "pregnancy_outcome_count"))
outcomes = (outcomes.groupby(cols + ['sub_entity'], observed=False)['value'].sum()
            / outcomes.groupby(cols, observed=False)['value'].sum()).reset_index()
outcomes = outcomes.loc[outcomes.scenario=='baseline']
outcomes = outcomes.groupby([c for c in cols if c != 'input_draw']
                           + ['sub_entity'], observed=False)['value'].describe(percentiles=[0.025,0.975]).reset_index()
outcomes.head()

In [ ]:
import matplotlib.pyplot as plt

p = outcomes.loc[outcomes.sub_entity=='live_birth']
plt.plot(p.age_group, p['mean'], marker='o', label='Simulation')
plt.fill_between(p.age_group, p['2.5%'], p['97.5%'], alpha=0.2)

plt.plot(p.age_group.iloc[:9], frac_live.reset_index().val, marker='o', label='GBD')
plt.plot(p.age_group.iloc[:9], art_live_frac, marker='o', label='Artifact')
plt.title('Live birth fraction')
plt.xticks(rotation=90)
plt.legend()
plt.ylim(0, 1)
plt.grid()

# don't remember why the artifact and GBD differ here...

In [ ]:
p = outcomes.loc[outcomes.sub_entity=='stillbirth']
plt.plot(p.age_group, p['mean'], marker='o', label='Simulation')
plt.fill_between(p.age_group, p['2.5%'], p['97.5%'], alpha=0.2)

plt.plot(p.age_group.iloc[:9], frac_still.val, marker='o', label='GBD')
plt.plot(p.age_group.iloc[:9], art_still_frac, marker='o', label='Artifact')
plt.title('Stillbirth fraction')
plt.xticks(rotation=90)
plt.legend()
plt.ylim(ymin=0)
plt.grid()

In [ ]:
p = outcomes.loc[outcomes.sub_entity=='partial_term']
plt.plot(p.age_group, p['mean'], marker='o', label='Simulation')
plt.fill_between(p.age_group, p['2.5%'], p['97.5%'], alpha=0.2)

plt.plot(p.age_group.iloc[:9], frac_other.val, marker='o', label='GBD')
plt.plot(p.age_group.iloc[:9], art_other_frac, marker='o', label='Artifact')
plt.title('Partial term fraction')
plt.xticks(rotation=90)
plt.legend()
plt.ylim(ymin=0)
plt.grid()

In [ ]:
pt = pd.read_parquet(paths.measure_path(results_dir, "person_time_pregnancy"))
pt.head()

In [ ]:
preg_transitions = pd.read_parquet(paths.measure_path(results_dir, "transition_count_pregnancy"))
preg_transitions.head()

In [ ]:
assert pt.loc[pt.sub_entity=='not_pregnant']['value'].sum() == 0, "Non-zero person time in the pregnancy state, which is unexpected"

In [ ]:
preg_transitions.loc[preg_transitions.scenario=='baseline'].groupby('sub_entity', observed=False)['value'].sum()

# why are these not 100_000? because they don't include deaths

In [ ]:
deaths = pd.read_parquet(paths.measure_path(results_dir, "deaths"))
deaths.head()

In [ ]:
assert (preg_transitions.loc[preg_transitions.scenario=='baseline'].groupby('sub_entity', observed=False)['value'].sum()
+ deaths.loc[deaths.scenario=='baseline']['value'].sum()).parturition_to_postpartum == 10_000_000 * deaths.input_draw.nunique(), "Sum of pregnancy transitions and deaths does not equal expected number of simulants"

In [ ]:
assert (preg_transitions.loc[preg_transitions.sub_entity=='parturition_to_postpartum']['value'].sum() 
        == preg_transitions.loc[preg_transitions.sub_entity=='parturition_to_postpartum']['value'].sum()), "Transition counts from preg->parturition do not equal countrs from parturition to postpartum"
assert preg_transitions.loc[preg_transitions.sub_entity=='postpartum_to_not_pregnant']['value'].sum() == 0, "Non-zero transition counts to the not pregnant state"

In [ ]:
((pt.loc[pt.sub_entity.isin(['pregnant','parturition'])].groupby('input_draw', observed=False)['value'].sum() * 365.25/7) 
 / preg_transitions.loc[preg_transitions.sub_entity=='pregnant_to_parturition'].groupby(['input_draw'], observed=False)['value'].sum()).mean(numeric_only=True)
# average pregnancy duration in weeks
# uniform distribution 6-24 weeks for partial term pregnancies (average 15 weeks)
    # therefore, the weighted AVERAGE population pregnancy duration across 
    # full and partial pregnancy term lengths should be ~30 weeks (15 * 0.4 + 40 * 0.6)
    # which looks right here! Great

In [ ]:
## This values matches to NO preg model results very closely 

In [ ]:
assert np.all(np.isclose(pt.loc[pt.sub_entity=='parturition'].groupby('input_draw', observed=False)['value'].sum() *365.25/7
        / preg_transitions.loc[preg_transitions.sub_entity=='parturition_to_postpartum'].groupby(['input_draw'], observed=False)['value'].sum(), 1)), "Time spent in parturition state is different than one week"

assert np.all(pt.loc[pt.sub_entity=='postpartum'].groupby('input_draw', observed=False)['value'].sum()
 / pt.loc[pt.sub_entity=='parturition'].groupby('input_draw', observed=False)['value'].sum()), "Postpartum state duration different than expected value of six weeks. Note this test assumes that the duration of the parturition state is confirmed to be one week"

In [ ]:
pt_summary = (pt.loc[pt.scenario=='baseline'].groupby(['input_draw','sub_entity'], observed=False)['value'].sum() / 10_000_000 * 365 / 7).groupby(['sub_entity'], observed=False).mean()
pt_summary

In [ ]:
if deaths.entity.unique != ['maternal_disorders']:
    assert deaths.loc[deaths.entity!='maternal_disorders']['value'].sum() == 0, "Deaths due to causes other than maternal disorders"

In [ ]:
pt_frac = (pt.groupby(['input_draw','age_group'], observed=False)[['value']].sum() / pt.groupby('input_draw', observed=False)[['value']].sum()).reset_index()
pt_frac.head()

In [ ]:
pop = get_population(release_id=16,
                    location_id=location_id,
                    sex_id=2,
                    age_group_id=age_group_ids,
                    year_id=year_id).rename(columns={'population':'val'})
pop = pop.set_index(['location_id','age_group_id'])[['val']]
pop

In [ ]:
pregs = preg_inc * pop
preg_frac = pregs / pregs.groupby('location_id', observed=False).sum()
preg_frac

In [ ]:
import matplotlib.pyplot as plt
#for draw in pt_frac.input_draw.unique():
#    plt.plot(pt_frac.age.unique(), pt_frac.loc[pt_frac.input_draw==draw]['value'], 
#             alpha=0.5, color='tab:blue')
plt.plot(pt_frac.age_group.unique(), pt_frac.groupby('age_group', observed=False).mean()[['value']], 
         color='tab:blue', label='Simulation')
plt.plot(pt_frac.age_group.iloc[:10], preg_frac.val, color='tab:orange', label='GBD')
plt.xticks(rotation=90)
plt.title('Fraction of all pregnancies by age group')
plt.grid()
plt.legend()
plt.xlabel('Age group')
plt.ylabel('Fraction of all pregnancies')
plt.show()

# Ok now let's check child data

In [ ]:
child_df = pd.read_parquet(paths.measure_path(results_dir, "births"))
child_df.head()

In [ ]:
plt.hist(child_df.gestational_age, bins=41)
plt.title('Gestational age (weeks) distribution')
plt.grid()
plt.show()


In [ ]:
plt.hist(child_df.birth_weight, bins=50)
plt.title('Birth weight (grams) distribution')
plt.grid()
plt.show()


In [ ]:
art = artifact.Artifact(artifact_dir)
art_exposure = art.load(f'risk_factor.low_birth_weight_and_short_gestation.exposure')['draw_0'].reset_index()
art_exposure = art_exposure.loc[(art_exposure.sex=='Female')][['parameter']]
art_cats = art.load(f'risk_factor.low_birth_weight_and_short_gestation.categories')
art_exposure['bw_start'] = None
art_exposure['bw_end'] = None
art_exposure['ga_start'] = None
art_exposure['ga_end'] = None
for i in art_exposure.index:
    art_exposure.loc[i, 'bw_start'] = art_cats[art_exposure.parameter[i]].split('wks, [')[1].split(',')[0]
    art_exposure.loc[i, 'bw_end'] = art_cats[art_exposure.parameter[i]].split('wks, [')[1].split(', ')[1].split(')')[0]
    art_exposure.loc[i, 'ga_start'] = art_cats[art_exposure.parameter[i]].split('- [')[1].split(', ')[0]
    art_exposure.loc[i, 'ga_end'] = art_cats[art_exposure.parameter[i]].split('- [')[1].split(') wks')[0].split(', ')[1]
art_exposure

In [ ]:
exp = art.load('risk_factor.low_birth_weight_and_short_gestation.exposure')
exp = exp.reset_index()

# for now averaging across sexes because we don't stratify for child data
exp = exp.groupby('parameter', observed=False).mean(numeric_only=True)
exp

In [ ]:
import re
import pandas as pd

def extract_number(x):
    x = str(x)
    match = re.search(r'\d+', x)
    return int(match.group()) if match else None

art_exposure['ga_start_num'] = art_exposure['ga_start'].apply(extract_number)
art_exposure['ga_end_num'] = art_exposure['ga_end'].apply(extract_number)
art_exposure['bw_start_num'] = art_exposure['bw_start'].apply(extract_number)
art_exposure['bw_end_num'] = art_exposure['bw_end'].apply(extract_number)

In [ ]:
cats = pd.Series(np.nan, index=child_df.index, name='parameter', dtype=object)

for i in list(range(0, len(art_exposure))):
    cat = art_exposure.loc[i]

    in_cat = (
        (child_df.gestational_age >= cat.ga_start_num)
        & (child_df.gestational_age < cat.ga_end_num)
        & (child_df.birth_weight >= cat.bw_start_num)
        & (child_df.birth_weight < cat.bw_end_num)
    )

    cats[in_cat] = cat.parameter

cats

In [ ]:
child_df_w_cats = pd.concat([child_df, cats], axis=1)
child_df_w_cats

# I am seeing some GAs that are too low...
# they should at least minimize at 6 weeks GA so there is a change to get intervention
# but they are so few that I am not sure if it matters


# we also want sex in the outputs

In [ ]:
child_df_w_cats['exposure'] = 1
exposure = (child_df_w_cats.groupby(['input_draw','parameter'], observed=False).exposure.sum() / child_df_w_cats.groupby(['input_draw'], observed=False).exposure.sum()).reset_index()
exposure

In [ ]:
width = 0.4
for draw in exposure.input_draw.unique():
    p = exposure.loc[exposure.input_draw==draw].merge(exp[[f'draw_{draw}']], on='parameter')
    p = p.sort_values(by=f'draw_{draw}').reset_index()
    plt.figure(dpi=1000)
    plt.bar(p.index, p[f'draw_{draw}'], width=width, label='GBD')
    plt.bar(p.index + width, p.exposure, width=width, label='Sim')
    plt.grid()
    plt.legend()
    plt.title(f'Draw {draw}')
    plt.semilogy()

In [ ]:
art_data = art_exposure.merge(exp.reset_index(), on='parameter')
#for col in ['bw_start','bw_end','ga_start','ga_end']:
    #art_data[f'{col}'] = art_data[f'{col}'].astype(int)
art_data['bw_mid'] = (art_data.bw_end_num - art_data.bw_start_num) / 2 + art_data.bw_start_num
art_data['ga_mid'] = (art_data.ga_end_num - art_data.ga_start_num) / 2 + art_data.ga_start_num

art_data.head()

In [ ]:
for col in ['bw_start', 'bw_end', 'ga_start', 'ga_end']:
    art_data[col] = (
        art_data[col]
        .astype(str)
        .str.extract(r'(\d+)')[0]
        .astype(int)
    )

In [ ]:
art_data['bw_mid'] = (art_data.bw_end - art_data.bw_start) / 2 + art_data.bw_start
art_data['ga_mid'] = (art_data.ga_end - art_data.ga_start) / 2 + art_data.ga_start

In [ ]:
import seaborn as sns
def plot_exposure_heat_map(draw):
    p = (art_data[['parameter','ga_mid','bw_mid',f'draw_{draw}']]
            .merge(exposure.loc[exposure.input_draw==draw], on=['parameter'], how='left'))
    p = p.replace(np.nan, 0)
    p['diff'] = p[f'draw_{draw}'] - p['exposure']
    p['percent_diff'] = p['diff'] / p[f'draw_{draw}'] * 100

    s = p.pivot_table(index='ga_mid', columns='bw_mid', values=f'draw_{draw}').sort_index(ascending=False)
    s = s.replace(0,np.nan)
    s = np.log(s/(1-s))
    a = p.pivot_table(index='ga_mid', columns='bw_mid', values='percent_diff').sort_index(ascending=False)
    a = a.replace(0,np.nan)
    #a=np.log(a/(1-a))
    fig, (ax1, ax2) = plt.subplots(1,2, dpi=200)
    sns.heatmap(s, ax=ax1,annot=True,cmap="coolwarm",vmin=-11,vmax=-1.5)
    sns.heatmap(a, ax=ax2,annot=True,cmap="coolwarm",vmin=-20,vmax=20)
    ax1.set_title('Logit-transformed exposure\ndistribution from artifact')
    ax2.set_title('Simulation exposure relative\nto artifact percent difference')
    for ax in [ax1,ax2]:
        ax.set_xlabel('Birthweight midpoint (g)')
        ax.set_ylabel('Gestational age midpoint (weeks)')
    plt.subplots_adjust(right=1.8, top=1)    
    plt.suptitle(f'Draw {draw}',
                x=0.9, y=1.25)

In [ ]:
for draw in exposure.input_draw.unique():
    plot_exposure_heat_map(draw)

# some notes on the interpretation of these plots:
    # in the righthand subplot, we expect to see DARK red and blue on the edges of the distribution that have low overall exposure and are therefore subject to more stochastic uncertainty
        # if we are seeing dark colors (big differences) in the categories on the right subplot for the categories with higher exposure (darker red in the lefthand plot), thise would be unexpected

    # we also do not want to see any noticable trend/pattern in color (blue versus red) that would indicate a systematic over or underestimation 

## Added V&V: denominator and linkage sanity

Cheap guards against the "empty run succeeds" failure class. The population-size assert
earlier in this notebook covers totals; these check per-seed plausibility and that the
births line list (the child sim's input) agrees with the outcome counts.

In [ ]:
outcomes = pd.read_parquet(paths.measure_path(results_dir, "pregnancy_outcome_count"))
per_seed = outcomes.groupby(["scenario", "input_draw", "random_seed"], observed=False).value.sum()
assert ((per_seed > 1_000) & (per_seed < 1_000_000)).all(), per_seed.describe()

lb = outcomes[outcomes.sub_entity == "live_birth"].groupby("scenario", observed=False).value.sum()
line_list = pd.read_parquet(paths.measure_path(results_dir, "births"))
ll = line_list[line_list.pregnancy_outcome == "live_birth"].groupby("scenario", observed=False).size()
assert (lb == ll).all(), pd.DataFrame({"outcome_count": lb, "line_list": ll})
print(f"outcomes per seed: {int(per_seed.min()):,} - {int(per_seed.max()):,} | "
      f"live births match the line list in all scenarios ({int(lb.sum()):,} total)")